In [4]:
import os
import pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score

HOME      = os.path.expanduser("~")
PRED_BASE = os.path.join(HOME, "methylbert_finetune/predictions")
DMR_SETS  = [250, 500, 1000]
VARIANTS  = [("M-aware", "w"), ("Ablation", "wo")]   # single seed (s42) for these sets

def test_metrics(n, wtag):
    path = os.path.join(PRED_BASE, f"predictions{n}",
                        f"preds_dmr{n}_s42_{wtag}_methylation_test.csv")
    df = pd.read_csv(path)                                   # comma-separated
    y    = df["y_oriented"].astype(int).values
    p    = df["p_oriented"].astype(float).values
    pred = (p > 0.5).astype(int)
    return (roc_auc_score(y, p),
            accuracy_score(y, pred),
            balanced_accuracy_score(y, pred))

res = {}
for n in DMR_SETS:
    for label, wtag in VARIANTS:
        res[(n, label)] = test_metrics(n, wtag)
        a, ac, b = res[(n, label)]
        print(f"{n:>4} {label:<9} AUROC {a:.4f}  Acc {ac:.4f}  Bal {b:.4f}")

print("\n% --- LaTeX table body ---")
print(r"\begin{tabular}{@{}r l rrr@{}}")
print(r"\toprule")
print(r"\textbf{DMRs} & \textbf{Model} & \textbf{AUROC} & \textbf{Acc.} & \textbf{Bal. acc.} \\")
print(r"\midrule")
for i, n in enumerate(DMR_SETS):
    if i > 0:
        print(r"\addlinespace")
    a, ac, b = res[(n, "M-aware")]
    print(f"{n}  & M-aware   & {a:.3f} & {ac:.3f} & {b:.3f} \\\\")
    a, ac, b = res[(n, "Ablation")]
    print(f"{n}  & Ablation  & {a:.3f} & {ac:.3f} & {b:.3f} \\\\")
    print(r"     & Two-means & TODO & TODO & TODO \\")
print(r"\bottomrule")
print(r"\end{tabular}")

 250 M-aware   AUROC 0.9835  Acc 0.9776  Bal 0.9776
 250 Ablation  AUROC 0.9858  Acc 0.9739  Bal 0.9739
 500 M-aware   AUROC 0.9856  Acc 0.9689  Bal 0.9687
 500 Ablation  AUROC 0.9841  Acc 0.9616  Bal 0.9617
1000 M-aware   AUROC 0.9825  Acc 0.9622  Bal 0.9622
1000 Ablation  AUROC 0.9847  Acc 0.9595  Bal 0.9595

% --- LaTeX table body ---
\begin{tabular}{@{}r l rrr@{}}
\toprule
\textbf{DMRs} & \textbf{Model} & \textbf{AUROC} & \textbf{Acc.} & \textbf{Bal. acc.} \\
\midrule
250  & M-aware   & 0.983 & 0.978 & 0.978 \\
250  & Ablation  & 0.986 & 0.974 & 0.974 \\
     & Two-means & TODO & TODO & TODO \\
\addlinespace
500  & M-aware   & 0.986 & 0.969 & 0.969 \\
500  & Ablation  & 0.984 & 0.962 & 0.962 \\
     & Two-means & TODO & TODO & TODO \\
\addlinespace
1000  & M-aware   & 0.983 & 0.962 & 0.962 \\
1000  & Ablation  & 0.985 & 0.960 & 0.960 \\
     & Two-means & TODO & TODO & TODO \\
\bottomrule
\end{tabular}


In [5]:
import os, sys
sys.path.insert(0, ".")                      # baseline module is in the notebook's dir
from baseline_two_means import compute_baseline

HOME      = os.path.expanduser("~")
DMR_SETS  = [250, 500, 1000]

base_rows = {}
for n in DMR_SETS:
    train = os.path.join(HOME, f"finetuneDatasets/dmr{n}/train_seq.csv")
    test  = os.path.join(HOME, f"finetuneTestdata/dmr{n}_test/data.csv")
    print(f"\n=== dmr{n} ===")
    r = compute_baseline(train, test, verbose=True)   # verbose prints the skip/balance diagnostics
    base_rows[n] = r

print("\n% --- Two-means rows for the sweep table ---")
for n in DMR_SETS:
    r = base_rows[n]
    print(f"{n}: Two-means & {r['auroc']:.3f} & {r['accuracy']:.3f} & {r['balanced_accuracy']:.3f} \\\\")


=== dmr250 ===
DMRs with train reads: 250 | both T&N: 250
windows dropped (no confident calls): train=0, eval=0
eval scored: 22667 | skipped (missing profile): 0
match-label balance: 50.1% positive
AUROC 0.9908 | acc 0.9763 | bal.acc 0.9763
confusion:
 [[11079   231]
 [  306 11051]]

=== dmr500 ===
DMRs with train reads: 500 | both T&N: 500
windows dropped (no confident calls): train=0, eval=0
eval scored: 44802 | skipped (missing profile): 0
match-label balance: 49.6% positive
AUROC 0.9895 | acc 0.9735 | bal.acc 0.9735
confusion:
 [[22069   520]
 [  668 21545]]

=== dmr1000 ===
DMRs with train reads: 1000 | both T&N: 1000
windows dropped (no confident calls): train=4, eval=5
eval scored: 88786 | skipped (missing profile): 0
match-label balance: 50.1% positive
AUROC 0.9879 | acc 0.9702 | bal.acc 0.9702
confusion:
 [[43188  1131]
 [ 1518 42949]]

% --- Two-means rows for the sweep table ---
250: Two-means & 0.991 & 0.976 & 0.976 \\
500: Two-means & 0.990 & 0.973 & 0.973 \\
1000: Two-me